# 🧱 Part 13: Mid-Training and Continued Pretraining

> **Previous context**: LoRA adapts a model with small trainable modules. Continued pretraining adapts by training on more domain text.
> **Goal for this part**: Understand CPT, domain adaptation, data mixing, and how loss curves reveal learning or forgetting.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Why continue pretraining

A general model may not know a specific domain well. Continued pretraining exposes it to domain text before downstream tuning.

## 1. Data mix

Pure domain data can improve specialization but may hurt general ability. Mixing general and domain data reduces forgetting.

## 2. Observation

Loss curves tell whether the model is adapting, overfitting, or forgetting. The point is not just to train longer, but to train on the right mixture.

## 3. CPT vs fine-tuning

CPT teaches language/domain distribution; instruction tuning teaches response behavior.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] CPT adapts a pretrained model to a domain.
- [ ] Data mixing controls specialization versus forgetting.
- [ ] Loss curves are diagnostic signals, not just scores.

Next, continue through the code cells for the Training Systems part and inspect the printed observations.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

In [ ]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

T = 1000
warmup = 50
stable_end = 850
eta_max = 0.01

def cosine_lr(t):
    'Read the values printed above and connect them to the concept in this cell.'
    if t < warmup:
        return eta_max * t / warmup
    progress = (t - warmup) / (T - warmup)
    return eta_max * 0.5 * (1 + np.cos(np.pi * progress))

def wsd_lr(t):
    'Read the values printed above and connect them to the concept in this cell.'
    if t < warmup:
        return eta_max * t / warmup
    elif t < stable_end:
        return eta_max  # Teaching note: follow this line to see the main step.
    else:
        progress = (t - stable_end) / (T - stable_end)
        return eta_max * (0.001 ** progress)

print('Read the values printed above and connect them to the concept in this cell.')
print("-" * 60)

for t in [0, 50, 200, 500, 700, 850, 900, 950, 999]:
    if t < warmup:
        phase = "Warmup"
    elif t < stable_end:
        phase = "Stable"
    else:
        phase = "Decay"
    
    cos = cosine_lr(t)
    wsd = wsd_lr(t)
    diff = 'Read the values printed above and connect them to the concept in this cell.' if cos > 0 else "—"
    print(f"{t:>6d}  {phase:>8s}  {cos:>12.6f}  {wsd:>12.6f}  {diff}")

print()
print('Key observation: inspect the values above and connect them to the idea in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Training')
print()
print('Conclusion: this small example shows the main trade-off.')

In [ ]:
# Teaching note: follow this line to see the main step.
steps = np.arange(T)
cos_lrs = [cosine_lr(t) for t in steps]
wsd_lrs = [wsd_lr(t) for t in steps]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Teaching note: follow this line to see the main step.
ax = axes[0]
ax.plot(steps, cos_lrs, 'b-', lw=1.5, label='Cosine')
ax.plot(steps, wsd_lrs, 'r-', lw=1.5, label='WSD')
ax.axvspan(0, warmup, alpha=0.1, color='green', label='Warmup')
ax.axvspan(warmup, stable_end, alpha=0.05, color='blue', label='Stable')
ax.axvspan(stable_end, T, alpha=0.1, color='red', label='Decay')
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine vs WSD')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Teaching note: follow this line to see the main step.
ax = axes[1]
ext = np.arange(600, 1200)
cos_ext = [cosine_lr(min(t, T-1)) for t in ext]  # Teaching note: follow this line to see the main step.
wsd_ext = [wsd_lr(t) if t < T else eta_max * (0.001 ** ((t - stable_end) / 200)) for t in ext]
ax.plot(ext, cos_ext, 'b-', lw=1.5, label='Cosine (continue training)')
ax.plot(ext, wsd_ext, 'r-', lw=1.5, label='WSD (continue training)')
ax.axvline(x=600, color='green', lw=1.5, label='Continuation start')
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Continuation: cosine is near zero')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
print('Loss')
print()

np.random.seed(42)
total_steps = 1000
stable_end = 850

steps = np.arange(total_steps)
loss = np.zeros(total_steps)

# Teaching note: follow this line to see the main step.
for t in range(stable_end):
    progress = (t + 1) / stable_end
    loss[t] = 3.0 * (progress ** (-0.05)) + np.random.normal(0, 0.02)

# Teaching note: follow this line to see the main step.
final_stable = loss[stable_end - 1]
for t in range(stable_end, total_steps):
    progress = (t - stable_end) / (total_steps - stable_end)
    drop = 0.15 * (1 - np.exp(-progress * 5))  # Teaching note: follow this line to see the main step.
    loss[t] = final_stable - drop + np.random.normal(0, 0.01)

# Teaching note: follow this line to see the main step.
stable_avg = np.mean(loss[800:850])
decay_avg = np.mean(loss[-50:])
improvement = stable_avg - decay_avg

stable_per_step = (4.0 - stable_avg) / stable_end
decay_per_step = improvement / (total_steps - stable_end)

print('Loss')
print('Loss')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
fig, axes = plt.subplots(2, 1, figsize=(10, 7))

# Teaching note: follow this line to see the main step.
ax = axes[0]
ax.plot(steps, loss, 'b-', lw=0.8, alpha=0.7)
ax.axvline(x=stable_end, color='red', ls='--', lw=1.5, label=f'Decay starts (step {stable_end})')
ax.axvspan(0, 50, alpha=0.1, color='green', label='Warmup')
ax.axvspan(50, stable_end, alpha=0.05, color='blue', label='Stable')
ax.axvspan(stable_end, 1000, alpha=0.1, color='red', label='Decay annealing')
ax.set_ylabel('Loss')
ax.set_title('WSD training loss')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Teaching note: follow this line to see the main step.
ax = axes[1]
ax.plot(steps[750:], loss[750:], 'b-', lw=1.5)
ax.axvline(x=stable_end, color='red', ls='--', lw=2, label='Decay starts')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Zoom: loss drops faster during annealing')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
decay_total = 200  # B tokens

schemes = [
    ('"no switch"',        100,  0,  0,  0),
    ('"conservative mix"',       70, 10, 10, 10),
    ('"balanced mix"',       50, 20, 15, 15),
    ('"aggressive mix"',       30, 30, 20, 20),
]

print('Read the values printed above and connect them to the concept in this cell.')
print("-" * 60)
for name, coarse, wiki, sft, inst in schemes:
    c = decay_total * coarse / 100
    w = decay_total * wiki / 100
    s = decay_total * sft / 100
    i = decay_total * inst / 100
    print(f"{name:<12s} {c:>6.0f}B  {w:>6.0f}B  {s:>6.0f}B  {i:>6.0f}B  ", end="")
    if coarse == 100:
        print("baseline")
    elif coarse >= 50:
        print('Read the values printed above and connect them to the concept in this cell.')
    else:
        print('Read the values printed above and connect them to the concept in this cell.')

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Training')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
fig, ax = plt.subplots(figsize=(8, 4))

names = ["'no switch'", "'conservative mix'", "'balanced mix'", "'aggressive mix'"]
coarse = [100, 70, 50, 30]
quality = [0, 30, 50, 70]  # Teaching note: follow this line to see the main step.

x = np.arange(len(names))
ax.bar(x, coarse, label='Coarse data (CC+Code)', color='#95a5a6', alpha=0.8)
ax.bar(x, quality, bottom=coarse, label='High-quality data (Wiki+SFT+instructions)', color='#2ecc71', alpha=0.8)
ax.set_ylabel('Share (%)')
ax.set_title('Data mix during decay')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.legend()
ax.axhline(y=50, color='red', ls='--', alpha=0.5, label='50% reference line')

# Teaching note: follow this line to see the main step.
ax.axvspan(0.5, 2.5, alpha=0.05, color='green')
ax.text(1.5, 55, 'Recommended range', ha='center', fontsize=10, color='green')

plt.tight_layout()
plt.show()


In [ ]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

cpt_total = 50  # B tokens

strategies = [
    ('Read the values printed above and connect them to the concept in this cell.',    100,   0, 'Read the values printed above and connect them to the concept in this cell.'),
    ('Read the values printed above and connect them to the concept in this cell.',       70,  30, 'Read the values printed above and connect them to the concept in this cell.'),
    ('"balanced mix"',       50,  50, 'Read the values printed above and connect them to the concept in this cell.'),
    ('"conservative mix"',       30,  70, 'Read the values printed above and connect them to the concept in this cell.'),
]

print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print("-" * 50)
for name, domain, general, note in strategies:
    d = cpt_total * domain / 100
    g = cpt_total * general / 100
    print(f"{name:<14s} {d:>6.0f}B  {g:>6.0f}B  {note}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
fig, ax = plt.subplots(figsize=(8, 4))

domain_pct = np.array([0, 30, 50, 70, 100])
forget_risk = np.array([0, 10, 25, 60, 90])  # Teaching note: follow this line to see the main step.
domain_learn = np.array([0, 40, 75, 90, 95])  # Teaching note: follow this line to see the main step.

ax2 = ax.twinx()
ax.plot(domain_pct, forget_risk, 'r-o', lw=2, ms=8, label='Forgetting risk')
ax2.plot(domain_pct, domain_learn, 'b--s', lw=2, ms=8, label='Domain learning')

ax.axvspan(40, 70, alpha=0.1, color='green', label='Recommended range')
ax.axvline(x=60, color='green', ls='--', alpha=0.7, label='Best ~60%')

ax.set_xlabel('Domain data share (%)')
ax.set_ylabel('Forgetting risk (%)', color='red')
ax2.set_ylabel('Domain learning effect (%)', color='blue')
ax.set_title('CPT sweet spot: 50-70% domain data')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='center right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Training')
print("-" * 60)

# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')

# LoRA CPT
print(f"{'LoRA CPT (r=16)':<20s} {'14 GB':>8s} {'~21M':>10s} {'~1 GB':>10s} {'~15 GB':>8s}")

# QLoRA CPT
print(f"{'QLoRA CPT (4bit)':<20s} {'3.5 GB':>8s} {'~21M':>10s} {'~1 GB':>10s} {'~6 GB':>8s}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')